In [1]:
import numpy as np
import pandas as pd

In [59]:
# Read only the column names (first row)
data2023 = pd.read_csv(fr"atusact-2023/atusact_2023.dat", dtype={'TRCODE': str, 'TUCASEID': str})
respondents2023 = pd.read_csv(fr"atusresp-2023/atusresp_2023.dat", usecols=['TUCASEID', 'TUDIARYDAY','TUFINLWGT'],dtype={'TUCASEID': str})
jointed_data2023 = pd.merge(data2023, respondents2023, on='TUCASEID', how='inner')
data2024 = pd.read_csv(fr"atusact-2024/atusact_2024.dat", dtype={'TRCODE': str, 'TUCASEID': str})
respondents2024 = pd.read_csv(fr"atusresp-2024/atusresp_2024.dat", usecols=['TUCASEID', 'TUDIARYDAY','TUFINLWGT'],dtype={'TUCASEID': str})
joined_data2024 = pd.merge(data2024, respondents2024, on='TUCASEID', how='inner')
joined_data = pd.concat([jointed_data2023, joined_data2024], ignore_index=True)

## Cooking

In [85]:
cooking = joined_data.where(joined_data['TRCODE'].astype(str) == '020201').dropna()
weight_sum = sum(cooking['TUFINLWGT'])
print(f"Total weight sum: {weight_sum}")

# 1. Setup Data (Assume 'cooking' is your dataframe)
# Ensure we are working with numeric duration
df = cooking[['TUACTDUR', 'TUFINLWGT']].copy()

# 2. Sort by the value you want to filter (Duration)
df = df.sort_values(by='TUACTDUR')

# 3. Calculate Cumulative Weights to find the true 98% Cutoff
cumsum_weights = df['TUFINLWGT'].cumsum()
total_weight = df['TUFINLWGT'].sum()
df['cumulative_percent'] = cumsum_weights / total_weight

# 4. Find the value at the 98th percentile mark
# We look for the first row where cumulative percent >= 0.98
cutoff_value = df.loc[df['cumulative_percent'] >= 0.98, 'TUACTDUR'].iloc[0]

print(f"98th Percentile Cutoff: {cutoff_value} minutes")

# 5. Filter the data (Keep everything <= cutoff)
clean_df = df[df['TUACTDUR'] <= cutoff_value].copy()

# 6. Calculate Weighted Stats on the CLEAN data
clean_values = clean_df['TUACTDUR']
clean_weights = clean_df['TUFINLWGT']

# Weighted Mean
weighted_mean = np.average(clean_values, weights=clean_weights)

# Weighted Std Dev
variance = np.average((clean_values - weighted_mean)**2, weights=clean_weights)
weighted_std = np.sqrt(variance)

print("-" * 30)
print(f"Original Count: {len(df)}")
print(f"Filtered Count: {len(clean_df)} (Removed top 2% of population)")
print(f"Clean Weighted Mean: {weighted_mean:.2f} min")
print(f"Clean Weighted Std:  {weighted_std:.2f} min")

Total weight sum: 196215413174.29675
98th Percentile Cutoff: 120.0 minutes
------------------------------
Original Count: 17231
Filtered Count: 17008 (Removed top 2% of population)
Clean Weighted Mean: 29.39 min
Clean Weighted Std:  24.88 min


In [89]:
# Convert time string to datetime and extract hour
cooking['start_hour'] = pd.to_datetime(cooking['TUSTARTTIM'], format='%H:%M:%S').dt.hour

# Define day type
cooking['day_type'] = cooking['TUDIARYDAY'].apply(lambda x: 'Weekend' if x in [1, 7] else 'Weekday')

# Define meal periods
def classify_meal(hour):
    if hour < 10:
        return 'Breakfast'
    elif hour < 15:
        return 'Lunch'
    else:
        return 'Dinner'

cooking['meal_period'] = cooking['start_hour'].apply(classify_meal)

# Function to calculate weighted stats with 98th percentile filtering
def calc_weighted_stats(df):
    if len(df) == 0:
        return None

    # Sort by duration for percentile calculation
    df_sorted = df.sort_values(by='TUACTDUR')

    # Calculate cumulative weights to find 98th percentile cutoff
    cumsum_weights = df_sorted['TUFINLWGT'].cumsum()
    total_weight = df_sorted['TUFINLWGT'].sum()
    df_sorted['cumulative_percent'] = cumsum_weights / total_weight

    # Find 98th percentile cutoff value
    cutoff_idx = df_sorted['cumulative_percent'] >= 0.98
    if cutoff_idx.any():
        cutoff_value = df_sorted.loc[cutoff_idx, 'TUACTDUR'].iloc[0]
    else:
        cutoff_value = df_sorted['TUACTDUR'].max()

    # Filter to remove top 2%
    df_clean = df_sorted[df_sorted['TUACTDUR'] <= cutoff_value].copy()

    # Calculate weighted statistics
    values = df_clean['TUACTDUR']
    weights = df_clean['TUFINLWGT']

    weighted_mean = np.average(values, weights=weights)
    variance = np.average((values - weighted_mean)**2, weights=weights)
    weighted_std = np.sqrt(variance)

    # Weighted median (approximate using cumulative distribution)
    cumsum = df_clean['TUFINLWGT'].cumsum()
    median_weight = cumsum.iloc[-1] / 2
    weighted_median = df_clean.loc[cumsum >= median_weight, 'TUACTDUR'].iloc[0]

    return {
        'count': len(df),
        'count_clean': len(df_clean),
        'total_weight': total_weight,
        'mean': weighted_mean,
        'std': weighted_std,
        'median': weighted_median,
        'cutoff_98': cutoff_value
    }

print("="*80)
print("COOKING DURATION ANALYSIS - WEIGHTED STATISTICS")
print("="*80)

# Overall statistics by meal period
print("\n" + "="*80)
print("BY MEAL PERIOD (ALL DAYS):")
print("="*80)

meal_stats = {}
for meal in ['Breakfast', 'Lunch', 'Dinner']:
    meal_df = cooking[cooking['meal_period'] == meal]
    stats = calc_weighted_stats(meal_df)
    meal_stats[meal] = stats

    if stats:
        print(f"\n{meal}:")
        print("-"*80)
        print(f"  Sample Count: {stats['count']} → {stats['count_clean']} (after filtering top 2%)")
        print(f"  Weighted Mean: {stats['mean']:.2f} min")
        print(f"  Weighted Std:  {stats['std']:.2f} min")
        print(f"  Weighted Median: {stats['median']:.2f} min")

# Statistics by day type and meal period
print("\n" + "="*80)
print("BY DAY TYPE AND MEAL PERIOD:")
print("="*80)

for day_type in ['Weekday', 'Weekend']:
    print(f"\n{day_type.upper()}:")
    print("="*80)

    for meal in ['Breakfast', 'Lunch', 'Dinner']:
        subset = cooking[(cooking['day_type'] == day_type) & (cooking['meal_period'] == meal)]
        stats = calc_weighted_stats(subset)

        if stats:
            print(f"\n  {meal}:")
            print("  " + "-"*76)
            print(f"    Sample Count: {stats['count']} → {stats['count_clean']} (after filtering)")
            print(f"    Weighted Mean: {stats['mean']:.2f} min")
            print(f"    Weighted Std:  {stats['std']:.2f} min")
            print(f"    Weighted Median: {stats['median']:.2f} min")

# Comparison table
print("\n" + "="*80)
print("SUMMARY COMPARISON TABLE:")
print("="*80)
print(f"{'Meal Period':<15} {'Day Type':<10} {'Mean (min)':<12} {'Std (min)':<12} {'Median (min)':<12}")
print("-"*80)

for meal in ['Breakfast', 'Lunch', 'Dinner']:
    for day_type in ['Weekday', 'Weekend']:
        subset = cooking[(cooking['day_type'] == day_type) & (cooking['meal_period'] == meal)]
        stats = calc_weighted_stats(subset)
        if stats:
            print(f"{meal:<15} {day_type:<10} {stats['mean']:>10.2f}   {stats['std']:>10.2f}   {stats['median']:>10.2f}")

COOKING DURATION ANALYSIS - WEIGHTED STATISTICS

BY MEAL PERIOD (ALL DAYS):

Breakfast:
--------------------------------------------------------------------------------
  Sample Count: 5880 → 5753 (after filtering top 2%)
  Weighted Mean: 19.24 min
  Weighted Std:  16.80 min
  Weighted Median: 15.00 min

Lunch:
--------------------------------------------------------------------------------
  Sample Count: 4065 → 3985 (after filtering top 2%)
  Weighted Mean: 30.75 min
  Weighted Std:  27.20 min
  Weighted Median: 20.00 min

Dinner:
--------------------------------------------------------------------------------
  Sample Count: 7286 → 7219 (after filtering top 2%)
  Weighted Mean: 36.15 min
  Weighted Std:  25.65 min
  Weighted Median: 30.00 min

BY DAY TYPE AND MEAL PERIOD:

WEEKDAY:

  Breakfast:
  ----------------------------------------------------------------------------
    Sample Count: 3117 → 3064 (after filtering)
    Weighted Mean: 17.95 min
    Weighted Std:  15.98 min
    W

In [87]:

bin_size = 5  # minutes

# ---------------------------------------------------------
# 1. PREPARE DATA & BINS
# ---------------------------------------------------------
# Convert time string to total minutes since midnight
cooking['start_minutes'] = pd.to_datetime(cooking['TUSTARTTIM'], format='%H:%M:%S').dt.hour * 60 + \
                           pd.to_datetime(cooking['TUSTARTTIM'], format='%H:%M:%S').dt.minute

# Group into bins
cooking['time_bin'] = (cooking['start_minutes'] // bin_size) * bin_size

# Define Day Type
cooking['day_type'] = cooking['TUDIARYDAY'].apply(lambda x: 'weekend' if x in [1, 7] else 'weekday')

# ---------------------------------------------------------
# 2. CALCULATE WEIGHTED VECTORS
# ---------------------------------------------------------

# Create a complete index of all possible bins (0 to 1435) to ensure no gaps (e.g., 3am might be empty)
all_bins = pd.DataFrame({'time_bin': range(0, 1440, bin_size)})

# Sum weights by bin and day_type
grouped = cooking.groupby(['time_bin', 'day_type'])['TUFINLWGT'].sum().reset_index()

# Pivot to separate columns (Weekday vs Weekend mass) and merge with full bin list
pivoted = grouped.pivot(index='time_bin', columns='day_type', values='TUFINLWGT').reset_index()
pivoted = pd.merge(all_bins, pivoted, on='time_bin', how='left').fillna(0)

# Calculate Grand Total Mass
grand_total_mass = cooking['TUFINLWGT'].sum()

# ---------------------------------------------------------
# 3. NORMALIZE TO PROBABILITIES (THE MATH)
# ---------------------------------------------------------

# Probability of event happening in this specific bin on ONE weekday
# (Volume_WD / Total) / 5 days
pivoted['prob_weekday'] = (pivoted['weekday'] / grand_total_mass) / 5.0

# Probability of event happening in this specific bin on ONE weekend day
# (Volume_WE / Total) / 2 days
pivoted['prob_weekend'] = (pivoted['weekend'] / grand_total_mass) / 2.0


In [88]:

# ---------------------------------------------------------
# 4. CONSTRUCT THE 7-DAY VECTOR (Mon -> Sun)
# ---------------------------------------------------------

# Define the week structure (Monday=0 to Sunday=6)
week_structure = [
    ('Monday', 'prob_weekday'),
    ('Tuesday', 'prob_weekday'),
    ('Wednesday', 'prob_weekday'),
    ('Thursday', 'prob_weekday'),
    ('Friday', 'prob_weekday'),
    ('Saturday', 'prob_weekend'),
    ('Sunday', 'prob_weekend')
]

full_week_list = []

for day_name, prob_col in week_structure:
    day_df = pivoted[['time_bin', prob_col]].copy()
    day_df.columns = ['minute_of_day', 'probability'] # Rename for consistency
    day_df['day_of_week'] = day_name
    
    # Create absolute time index for plotting/debugging
    # e.g., Monday 00:05, Tuesday 14:00
    day_df['time_label'] = day_df['minute_of_day'].apply(
        lambda m: f"{day_name} {int(m)//60:02d}:{int(m)%60:02d}"
    )
    
    full_week_list.append(day_df)

# Concatenate into one long list (Monday 00:00 -> Sunday 23:55)
final_week_pdf = pd.concat(full_week_list, ignore_index=True)

# ---------------------------------------------------------
# 5. VERIFICATION & EXPORT
# ---------------------------------------------------------

total_sum = final_week_pdf['probability'].sum()

print(f"Total Rows: {len(final_week_pdf)} (Should be 2016 for 5-min bins)")
print(f"Total Sum of Probabilities: {total_sum:.6f} (Should be 1.0)")

# Save
final_week_pdf.to_csv(f"../data/activity_initial_probability/cooking_start_time_{bin_size}min_bins.csv", index=False)

Total Rows: 2016 (Should be 2016 for 5-min bins)
Total Sum of Probabilities: 1.000000 (Should be 1.0)


## Laundry

In [82]:
laundry = joined_data.where(joined_data['TRCODE'].astype(str) == '020102').dropna()

In [84]:

bin_size = 5  # minutes

# ---------------------------------------------------------
# 1. PREPARE DATA & BINS
# ---------------------------------------------------------
# Convert time string to total minutes since midnight
laundry['start_minutes'] = pd.to_datetime(laundry['TUSTARTTIM'], format='%H:%M:%S').dt.hour * 60 + \
                           pd.to_datetime(laundry['TUSTARTTIM'], format='%H:%M:%S').dt.minute

# Group into bins
laundry['time_bin'] = (laundry['start_minutes'] // bin_size) * bin_size
# Define Day Type
laundry['day_type'] = laundry['TUDIARYDAY'].apply(lambda x: 'weekend' if x in [1, 7] else 'weekday')

# ---------------------------------------------------------
# 2. CALCULATE WEIGHTED VECTORS
# ---------------------------------------------------------

# Create a complete index of all possible bins (0 to 1435) to ensure no gaps (e.g., 3am might be empty)
all_bins = pd.DataFrame({'time_bin': range(0, 1440, bin_size)})

# Sum weights by bin and day_type
grouped = laundry.groupby(['time_bin', 'day_type'])['TUFINLWGT'].sum().reset_index()

# Pivot to separate columns (Weekday vs Weekend mass) and merge with full bin list
pivoted = grouped.pivot(index='time_bin', columns='day_type', values='TUFINLWGT').reset_index()
pivoted = pd.merge(all_bins, pivoted, on='time_bin', how='left').fillna(0)

# Calculate Grand Total Mass
grand_total_mass = laundry['TUFINLWGT'].sum()

# ---------------------------------------------------------
# 3. NORMALIZE TO PROBABILITIES (THE MATH)
# ---------------------------------------------------------

# Probability of event happening in this specific bin on ONE weekday
# (Volume_WD / Total) / 5 days
pivoted['prob_weekday'] = (pivoted['weekday'] / grand_total_mass) / 5.0

# Probability of event happening in this specific bin on ONE weekend day
# (Volume_WE / Total) / 2 days
pivoted['prob_weekend'] = (pivoted['weekend'] / grand_total_mass) / 2.0



# ---------------------------------------------------------
# 4. CONSTRUCT THE 7-DAY VECTOR (Mon -> Sun)
# ---------------------------------------------------------

# Define the week structure (Monday=0 to Sunday=6)
week_structure = [
    ('Monday', 'prob_weekday'),
    ('Tuesday', 'prob_weekday'),
    ('Wednesday', 'prob_weekday'),
    ('Thursday', 'prob_weekday'),
    ('Friday', 'prob_weekday'),
    ('Saturday', 'prob_weekend'),
    ('Sunday', 'prob_weekend')
]

full_week_list = []

for day_name, prob_col in week_structure:
    day_df = pivoted[['time_bin', prob_col]].copy()
    day_df.columns = ['minute_of_day', 'probability'] # Rename for consistency
    day_df['day_of_week'] = day_name
    
    # Create absolute time index for plotting/debugging
    # e.g., Monday 00:05, Tuesday 14:00
    day_df['time_label'] = day_df['minute_of_day'].apply(
        lambda m: f"{day_name} {int(m)//60:02d}:{int(m)%60:02d}"
    )
    
    full_week_list.append(day_df)

# Concatenate into one long list (Monday 00:00 -> Sunday 23:55)
final_week_pdf = pd.concat(full_week_list, ignore_index=True)

# ---------------------------------------------------------
# 5. VERIFICATION & EXPORT
# ---------------------------------------------------------

total_sum = final_week_pdf['probability'].sum()

print(f"Total Rows: {len(final_week_pdf)} (Should be 2016 for 5-min bins)")
print(f"Total Sum of Probabilities: {total_sum:.6f} (Should be 1.0)")

# Save
final_week_pdf.to_csv(f"../data/activity_initial_probability/laundry_start_time_{bin_size}min_bins.csv", index=False)


Total Rows: 2016 (Should be 2016 for 5-min bins)
Total Sum of Probabilities: 1.000000 (Should be 1.0)


## Dishwasher

In [91]:
cleanup = joined_data.where(joined_data['TRCODE'].astype(str) == '020203').dropna()


In [92]:

bin_size = 5  # minutes

# ---------------------------------------------------------
# 1. PREPARE DATA & BINS
# ---------------------------------------------------------
# Convert time string to total minutes since midnight
cleanup['start_minutes'] = pd.to_datetime(cleanup['TUSTARTTIM'], format='%H:%M:%S').dt.hour * 60 + \
                           pd.to_datetime(cleanup['TUSTARTTIM'], format='%H:%M:%S').dt.minute

# Group into bins
cleanup['time_bin'] = (cleanup['start_minutes'] // bin_size) * bin_size
# Define Day Type
cleanup['day_type'] = cleanup['TUDIARYDAY'].apply(lambda x: 'weekend' if x in [1, 7] else 'weekday')

# ---------------------------------------------------------
# 2. CALCULATE WEIGHTED VECTORS
# ---------------------------------------------------------

# Create a complete index of all possible bins (0 to 1435) to ensure no gaps (e.g., 3am might be empty)
all_bins = pd.DataFrame({'time_bin': range(0, 1440, bin_size)})

# Sum weights by bin and day_type
grouped = cleanup.groupby(['time_bin', 'day_type'])['TUFINLWGT'].sum().reset_index()

# Pivot to separate columns (Weekday vs Weekend mass) and merge with full bin list
pivoted = grouped.pivot(index='time_bin', columns='day_type', values='TUFINLWGT').reset_index()
pivoted = pd.merge(all_bins, pivoted, on='time_bin', how='left').fillna(0)

# Calculate Grand Total Mass
grand_total_mass = cleanup['TUFINLWGT'].sum()

# ---------------------------------------------------------
# 3. NORMALIZE TO PROBABILITIES (THE MATH)
# ---------------------------------------------------------

# Probability of event happening in this specific bin on ONE weekday
# (Volume_WD / Total) / 5 days
pivoted['prob_weekday'] = (pivoted['weekday'] / grand_total_mass) / 5.0

# Probability of event happening in this specific bin on ONE weekend day
# (Volume_WE / Total) / 2 days
pivoted['prob_weekend'] = (pivoted['weekend'] / grand_total_mass) / 2.0



# ---------------------------------------------------------
# 4. CONSTRUCT THE 7-DAY VECTOR (Mon -> Sun)
# ---------------------------------------------------------

# Define the week structure (Monday=0 to Sunday=6)
week_structure = [
    ('Monday', 'prob_weekday'),
    ('Tuesday', 'prob_weekday'),
    ('Wednesday', 'prob_weekday'),
    ('Thursday', 'prob_weekday'),
    ('Friday', 'prob_weekday'),
    ('Saturday', 'prob_weekend'),
    ('Sunday', 'prob_weekend')
]

full_week_list = []

for day_name, prob_col in week_structure:
    day_df = pivoted[['time_bin', prob_col]].copy()
    day_df.columns = ['minute_of_day', 'probability'] # Rename for consistency
    day_df['day_of_week'] = day_name
    
    # Create absolute time index for plotting/debugging
    # e.g., Monday 00:05, Tuesday 14:00
    day_df['time_label'] = day_df['minute_of_day'].apply(
        lambda m: f"{day_name} {int(m)//60:02d}:{int(m)%60:02d}"
    )
    
    full_week_list.append(day_df)

# Concatenate into one long list (Monday 00:00 -> Sunday 23:55)
final_week_pdf = pd.concat(full_week_list, ignore_index=True)

# ---------------------------------------------------------
# 5. VERIFICATION & EXPORT
# ---------------------------------------------------------

total_sum = final_week_pdf['probability'].sum()

print(f"Total Rows: {len(final_week_pdf)} (Should be 2016 for 5-min bins)")
print(f"Total Sum of Probabilities: {total_sum:.6f} (Should be 1.0)")

# Save
final_week_pdf.to_csv(f"../data/activity_initial_probability/dishwasher_start_time_{bin_size}min_bins.csv", index=False)

Total Rows: 2016 (Should be 2016 for 5-min bins)
Total Sum of Probabilities: 1.000000 (Should be 1.0)
